# HoloViews の基本

HoloViews は、ほかの描画ライブラリの上に乗る、可視化と分析のためのライブラリです。データに「どの種類の図か」「どの列がどの次元か」といった注釈を付けると、図にしてくれます。実際の描画は matplotlib や Bokeh が行います。

Notebook で使うときは、まず `extension()` で描画に使うライブラリを選びます。Bokeh なら次のとおりです。

```python
hv.extension('bokeh')
```

この章では、HoloViews で可視化のアプリを作る方法を説明します。あわせて Param と Panel も紹介します。この 2 つと HoloViews を組み合わせると、対話的なアプリが簡単に作れます。

* `Element` と `Dimension`
* 整然データからの `Dataset`
* コンテナ(`HoloMap`、`NdLayout`、`NdOverlay`、`Layout`、`Overlay`)
* プロットのオプション(`opts`)

In [1]:
import pprint
import holoviews as hv
from holoviews import streams
import numpy as np
import pandas as pd
from holoviews import opts
import panel as pn

hv.extension("bokeh", inline=False, logo=False)
pn.extension()

## `Element` オブジェクト

HoloViews では、`Element` の派生クラスでデータに注釈を付けます。次は `Element` の派生の木です。曲線の `Curve` は `Chart` から、`Chart` は `Dataset` と `Element2D` から派生しています。

In [1]:
from helper.python import print_subclasses

print_subclasses(hv.Element, as_html=True, height=300)


`Element` は、いろいろな形のデータを受け取れます。次のプログラムは、タプル、`DataFrame`、`ndarray`、辞書の 4 つを作り、それぞれを `Curve` で包みます。区別しやすいように `label` を変え、`data` の型を調べます。`c1` と `c2` は `DataFrame` に、`c3` は `ndarray`、`c4` は `OrderedDict` になります。タプルは `DataFrame` に変換されます。

In [3]:
x = np.linspace(0, 4 * np.pi, 200)
y = np.sin(x)

tuple_data = (x, y)
df_data = pd.DataFrame(dict(x=x, y=y))
arr_data = np.c_[x, y]
dict_data = dict(x=x, y=y)

c1 = hv.Curve(tuple_data, label="c1")
c2 = hv.Curve(df_data, label="c2")
c3 = hv.Curve(arr_data, label="c3")
c4 = hv.Curve(dict_data, label="c4")

curves = [c1, c2, c3, c4]
print(" ".join(f"{c.label}: {type(c.data).__name__}" for c in curves))

c1: DataFrame c2: DataFrame c3: ndarray c4: OrderedDict


型によって、データの持ち方(インターフェース)が違います。`interface` 属性で分かります。

In [4]:
print(" ".join(f"{c.label}: {c.interface.__name__}" for c in curves))

c1: PandasInterface c2: PandasInterface c3: ArrayInterface c4: DictInterface


`c3` は配列で持っていますが、`dframe()` で `DataFrame` に変換できます。

In [5]:
print(c3.dframe().head())

          x         y
0  0.000000  0.000000
1  0.063148  0.063106
2  0.126295  0.125960
3  0.189443  0.188312
4  0.252590  0.249913


HoloViews のクラスは `param.Parameterized` から派生しているので、`.param.values()` で属性を全部見られます。`kdims`、`vdims`、`group`、`label` の 4 つが特に大事です。

In [6]:
c4.param.values()

{'cdims': {},
 'datatype': ['dataframe',
  'dictionary',
  'grid',
  'xarray',
  'multitabular',
  'spatialpandas',
  'dask_spatialpandas',
  'dask',
  'cuDF',
  'array',
  'narwhals',
  'ibis'],
 'extents': (None, None, None, None),
 'group': 'Curve',
 'kdims': [Dimension('x')],
 'label': 'c4',
 'name': 'Curve00278',
 'vdims': [Dimension('y')]}

:::{tip}
Notebook では、セルの最後の式が表示できるオブジェクトなら、選んだライブラリで図になります。`display()` でも出せます。

```python
from IPython.display import display
display(c1)
```
:::

## `Dimension` オブジェクト

次元の情報は `Dimension` で持ちます。属性は次のとおりです。

| 属性 | 型 | 既定値 | 説明 |
|:--|:--|:--|:--|
| `name` | String | `'Dimension'` | 次元の名前。Python の識別子にする |
| `label` | String | `None` | 表示用の名前。LaTeX も使える |
| `cyclic` | Boolean | `False` | 値が周期的か(角度など) |
| `value_format` | Callable | `None` | 値の書式にする関数 |
| `range` | Tuple | `(None, None)` | 値の範囲 |
| `soft_range` | Tuple | `(None, None)` | 目安の範囲 |
| `type` | Parameter | `None` | 値の型 |
| `default` | Parameter | `None` | 既定値 |
| `step` | Number | `None` | スライダーの刻み |
| `unit` | String | `None` | 単位 |
| `values` | List | `[]` | 選べる値の一覧 |

`Element` は、キー次元のリスト `kdims` と、値次元のリスト `vdims` を持ちます。どちらも `Dimension` です。種類によって、この次元をどう図にするかが違います。たとえば `Curve` は、キー次元 `"x"` が各点の x 座標、値次元 `"y"` が y 座標です。

In [7]:
print(hv.Curve.kdims, hv.Curve.vdims)

[Dimension('x')] [Dimension('y')]


線分をまとめた `Segments` は、キー次元が 4 つで、両端の x と y に対応します。値次元はありません。

In [8]:
print(hv.Segments.kdims, hv.Segments.vdims)

[Dimension('x0'), Dimension('y0'), Dimension('x1'), Dimension('y1')] []


箱ひげ図の `BoxWhisker` は、キー次元がなく、値次元が 1 つです。

In [9]:
print(hv.BoxWhisker.kdims, hv.BoxWhisker.vdims)

[] [Dimension('y')]


`Element` を作るときに、`kdims` と `vdims` で次元を決められます。次は 3 通りの書き方です。`c5` は文字列で、`Dimension` の `name` と `label` を同時に決めたのと同じです。`c6` は 2 つのタプルで、`name` と `label` を別々に決めます。`c7` は `Dimension` をそのまま渡すので、キーワードで全部の属性を決められます。リストでないものを渡すと、長さ 1 のリストを渡したのと同じになります。

In [10]:
c5 = hv.Curve(tuple_data, kdims="time", vdims="position", label="c5")
c6 = hv.Curve(tuple_data, kdims=("t", "Time"), vdims=("x", "Position"), label="c6")
c7 = hv.Curve(
    tuple_data,
    label="c7",
    kdims=hv.Dimension("t", label="Time", unit="s"),
    vdims=hv.Dimension("x", label="Position", unit="mm"),
)

for c in (c5, c6, c7):
    kdim = c.kdims[0]
    vdim = c.vdims[0]
    print(f"{c.label}: {kdim.name=} {kdim.label=} {kdim.unit=} ")

c5: kdim.name='time' kdim.label='time' kdim.unit=None 
c6: kdim.name='t' kdim.label='Time' kdim.unit=None 
c7: kdim.name='t' kdim.label='Time' kdim.unit='s' 


図にすると、x 軸の題はキー次元の `label` と `unit`、y 軸の題は値次元の `label` と `unit` で決まります。たとえば `c7` は次のようになります。

In [11]:
c7.opts(width=600, show_grid=True)

:Curve   [t]   (Position)

次元の情報を変えたいとき、属性を直接書き換えることもできますが、`redim` を使うのが勧めです。元のオブジェクトは変えず、新しいオブジェクトを返します。`redim()` で次元そのものを変えられます。

❶ `c1` の次元は `x` と `y` です。`redim()` のキーワードで決めます。`x` には文字列を渡すと、その名前の `Dimension` を作ります。`y` には `Dimension` をそのまま渡します。

`redim` はメソッドではなく、属性をいじるためのオブジェクトです。❷ `redim.unit()` は `unit`、❸ `redim.range()` は `range` を変えます。`range` を決めると、軸の表示範囲がその値になり、データの範囲ではなくなります。

In [12]:
c8 = c1.redim(x="time", y=hv.Dimension("position", label="Position"))  # ❶
c9 = c5.redim.unit(time="s", position="mm")  # ❷
c10 = c7.redim.range(t=(0, 8), x=(-2, 2))  # ❸

print(f"{c7.kdims[0].range = }, {c10.kdims[0].range = }")

c7.kdims[0].range = (None, None), c10.kdims[0].range = (0, 8)


## `Dataset` から図へ

データが整っているなら、`Element` を作って次元を決めるのは面倒です。整然データ(tidy data)からは、もっと簡単に図を作れます。

2 次元の表では、列が次元、行が観測です。列は、自分で決める独立変数と、測って得る従属変数に分かれます。次のプログラムは `gapminder_CJU.csv` を読んで `df` にします。`country` が国、`year` が年、`gdp_capita`、`life`、`population`、`ppp` などが値です。

In [13]:
import pandas as pd

df = pd.read_csv("data/gapminder_CJU.csv")
df.sample(5).sort_index()

,country,year,gdp_capita,life,ppp_capita,population
1,China,1991,786.0,68.8,1090.0,1190000000
29,Japan,1990,38100.0,79.3,19600.0,125000000
38,Japan,1999,41100.0,81.0,25300.0,127000000
53,Japan,2014,46500.0,83.8,39200.0,128000000
59,United States,1991,35500.0,75.7,24300.0,255000000


`df` から `Dataset` を作り、`country` と `year` をキー次元にします。残りの列は値次元になります。表示では、キー次元が角括弧、値次元が丸括弧に入ります。

In [14]:
ds = hv.Dataset(df, kdims=["country", "year"])
ds

:Dataset   [country,year]   (gdp_capita,life,ppp_capita,population)

`ds.to.curve()` で、`Dataset` を `Curve` のコンテナに変えます。`kdims` と `vdims` で次元を決めます。`country` を指定しないので、グループの次元になります。結果は `HoloMap` で、キー次元が `country` です。`print()` で中身を見ると、`hm` の各値が `Curve`(キーが `year`、値が `ppp_capita`)だと分かります。

In [15]:
hm = ds.to.curve(kdims="year", vdims="ppp_capita")
print(hm)

:HoloMap   [country]
   :Curve   [year]   (ppp_capita)


`HoloMap` は `UniformNdMapping` から派生しています。次の図は、その派生クラスです。どれも複数の要素を持つ入れ物で、見せ方が違うだけで、データの持ち方は同じです。

In [2]:
print_subclasses(hv.core.UniformNdMapping, as_html=True, height=300)


各オブジェクトの `kdims` と `vdims` を見ます。`HoloMap` は添字で要素を取れるので、`hm['China']` で `country` が `'China'` の要素が取れます。

In [17]:
print(f"{hm.kdims = }")
print(f'{hm["China"].kdims = }')
print(f'{hm["China"].vdims = }')

hm.kdims = [Dimension('country')]
hm["China"].kdims = [Dimension('year')]
hm["China"].vdims = [Dimension('ppp_capita')]


`data` 属性が実際のデータです。`HoloMap` は順序付き辞書、`Curve` は `DataFrame` で持ちます。

In [18]:
print("HoloMap data:")
pprint.pp(hm.data)
print()
print("Curve data:")
print(hm["China"].data.head())

HoloMap data:
{('China',): :Curve   [year]   (ppp_capita),
 ('Japan',): :Curve   [year]   (ppp_capita),
 ('United States',): :Curve   [year]   (ppp_capita)}

Curve data:
  country  year  gdp_capita  life  ppp_capita  population
0   China  1990       729.0  68.7       982.0  1180000000
1   China  1991       786.0  68.8      1090.0  1190000000
2   China  1992       887.0  69.1      1260.0  1210000000
3   China  1993       998.0  69.4      1460.0  1220000000
4   China  1994      1120.0  69.7      1660.0  1230000000


`hm` をそのまま出すと、左の図のようにドロップダウンで `country` を選べます。選ぶと曲線が入れ替わります。

In [19]:
hm.opts(width=400)

:HoloMap   [country]
   :Curve   [year]   (ppp_capita)

複数の曲線を同時に出したいときは、`overlay()` で `NdOverlay` にします。`NdOverlay` は要素を重ねて描き、キー次元の値が凡例になります。

In [20]:
year_ppp_plot = hm.overlay()
print(year_ppp_plot)
year_ppp_plot

:NdOverlay   [country]
   :Curve   [year]   (ppp_capita)


:NdOverlay   [country]
   :Curve   [year]   (ppp_capita)

次の例では、国ごとに分け、各要素を `Scatter` にします。キー次元が `ppp_capita`、値次元が `life` と `year` です。`Scatter` はキー次元を x 軸、1 つ目の値次元(`life`)を y 軸にします。`hover` ツールを足しているので、点にマウスを置くと、使っていない `year` の値も見られます。

In [21]:
gdp_life_plot = ds.to.scatter(
    kdims=["ppp_capita"], vdims=["life", "year"], groupby=["country"]
).overlay()
gdp_life_plot.opts(width=400)

:NdOverlay   [country]
   :Scatter   [ppp_capita]   (life,year)

`HoloMap.layout()` で `NdLayout` にすると、複数の図を並べて出せます。

In [22]:
hm.layout()

:NdLayout   [country]
   :Curve   [year]   (ppp_capita)

## 並べる・重ねる入れ物

`NdLayout` と `NdOverlay` は、多次元の入れ物で、並べるか重ねるかで図をまとめます。次元のない `Layout` と `Overlay` もあります。これらは `+` と `*` の演算子で作れます。`+` は並べる(`Layout`)、`*` は重ねる(`Overlay`)です。次のコードは、1 本の曲線と点を重ねます。

```python
hv.Curve((x1, y1)) * hv.Scatter((x2, y2))
```

次の `orth_poly()` は、曲線とその極値を作ります。まず点の座標を計算し、`scipy.signal.find_peaks()` で極大と極小の位置を求めます。

❶ `Curve` を作り、`group` と `label` を付けます。同じ入れ物の中で要素を区別するのに使います。`label` は凡例に出ます。❷ 極値の座標で `Scatter` を作り、こちらは別の `group` にします。❸ 値の数字を出す `Labels` を 2 つ作ります。

In [23]:
import numpy as np
from scipy import special
from scipy import signal


def orth_poly(f, n, x0=-1, x1=1):
    x = np.linspace(x0, x1, 100)
    y = f(n, x)

    index_hi, _ = signal.find_peaks(y)
    index_lo, _ = signal.find_peaks(-y)
    name = f.__name__.split("_")[-1]
    label = f"{name} {n}"
    curve = hv.Curve((x, y), group="curve", label=label)  # ❶

    index = np.r_[index_hi, index_lo]
    xp = x[index]
    yp = y[index]
    peak = hv.Scatter((xp, yp), group="peak", label=label)  # ❷

    xh = x[index_hi]
    yh = y[index_hi]
    texth = ["{:4.2f}".format(v) for v in yh]
    text_hi = hv.Labels((xh, yh, texth), group="text_hi", label=label)  # ❸

    xl = x[index_lo]
    yl = y[index_lo]
    textl = ["{:4.2f}".format(v) for v in yl]
    text_lo = hv.Labels((xl, yl, textl), group="text_lo", label=label)  # ❸

    return curve * peak * text_hi * text_lo  # ❹

`orth_poly()` で 4 次のチェビシェフ多項式の図 `curve1` を作ります。これは `Overlay` で、4 つの要素を `OrderedDict` で持ちます。キーは `group` と `label` から作られます。キーの文字が大文字で始まるのに注意してください。

In [24]:
from pprint import pp

curve1 = orth_poly(special.eval_chebyu, 4)
print(curve1)
print()
pp(curve1.data)

:Overlay
   .Curve.Chebyu_4   :Curve   [x]   (y)
   .Peak.Chebyu_4    :Scatter   [x]   (y)
   .Text_hi.Chebyu_4 :Labels   [x,y]   (Label)
   .Text_lo.Chebyu_4 :Labels   [x,y]   (Label)

{('Curve', 'Chebyu_4'): :Curve   [x]   (y),
 ('Peak', 'Chebyu_4'): :Scatter   [x]   (y),
 ('Text_hi', 'Chebyu_4'): :Labels   [x,y]   (Label),
 ('Text_lo', 'Chebyu_4'): :Labels   [x,y]   (Label)}


属性と同じように要素を取り出せます。1 段目の属性で `group` が同じものをまとめて取り、2 段目で 1 つの要素を取ります。

In [25]:
print(curve1.Curve)
print()
print(curve1.Curve.Chebyu_4)

:Overlay
   .Chebyu_4 :Curve   [x]   (y)

:Curve   [x]   (y)


要素の数が決まらないときは、`Overlay` を直接使います。次のコードは `orth_poly()` をループで呼び、3 つの次数のチェビシェフ多項式を `hv.Overlay()` で重ねます。12 個の要素を持つ `Overlay` ができます。`chebyc_plot.Peak` で、`group` が `peak` の要素だけを取り出せます。

In [26]:
chebyc_plot = hv.Overlay([orth_poly(special.eval_chebyu, n) for n in [2, 3, 4]])
print(chebyc_plot)
print()
print(chebyc_plot.Peak)

:Overlay
   .Curve.Chebyu_2   :Curve   [x]   (y)
   .Peak.Chebyu_2    :Scatter   [x]   (y)
   .Text_hi.Chebyu_2 :Labels   [x,y]   (Label)
   .Text_lo.Chebyu_2 :Labels   [x,y]   (Label)
   .Curve.Chebyu_3   :Curve   [x]   (y)
   .Peak.Chebyu_3    :Scatter   [x]   (y)
   .Text_hi.Chebyu_3 :Labels   [x,y]   (Label)
   .Text_lo.Chebyu_3 :Labels   [x,y]   (Label)
   .Curve.Chebyu_4   :Curve   [x]   (y)
   .Peak.Chebyu_4    :Scatter   [x]   (y)
   .Text_hi.Chebyu_4 :Labels   [x,y]   (Label)
   .Text_lo.Chebyu_4 :Labels   [x,y]   (Label)

:Overlay
   .Chebyu_2 :Scatter   [x]   (y)
   .Chebyu_3 :Scatter   [x]   (y)
   .Chebyu_4 :Scatter   [x]   (y)


同じやり方で、3 つの次数のルジャンドル多項式の `legendre_plot` を作り、`+` で 2 つを並べます。結果は `Layout` です。`group` と `label` を付けていないので、HoloViews がキーを付けます。

In [27]:
legendre_plot = hv.Overlay([orth_poly(special.eval_legendre, n) for n in [2, 3, 4]])
plots = chebyc_plot + legendre_plot
print(plots)

:Layout
   .Overlay.I  :Overlay
      .Curve.Chebyu_2   :Curve   [x]   (y)
      .Peak.Chebyu_2    :Scatter   [x]   (y)
      .Text_hi.Chebyu_2 :Labels   [x,y]   (Label)
      .Text_lo.Chebyu_2 :Labels   [x,y]   (Label)
      .Curve.Chebyu_3   :Curve   [x]   (y)
      .Peak.Chebyu_3    :Scatter   [x]   (y)
      .Text_hi.Chebyu_3 :Labels   [x,y]   (Label)
      .Text_lo.Chebyu_3 :Labels   [x,y]   (Label)
      .Curve.Chebyu_4   :Curve   [x]   (y)
      .Peak.Chebyu_4    :Scatter   [x]   (y)
      .Text_hi.Chebyu_4 :Labels   [x,y]   (Label)
      .Text_lo.Chebyu_4 :Labels   [x,y]   (Label)
   .Overlay.II :Overlay
      .Curve.Legendre_2   :Curve   [x]   (y)
      .Peak.Legendre_2    :Scatter   [x]   (y)
      .Text_hi.Legendre_2 :Labels   [x,y]   (Label)
      .Text_lo.Legendre_2 :Labels   [x,y]   (Label)
      .Curve.Legendre_3   :Curve   [x]   (y)
      .Peak.Legendre_3    :Scatter   [x]   (y)
      .Text_hi.Legendre_3 :Labels   [x,y]   (Label)
      .Text_lo.Legendre_3 :Labels   [x,y

何段かの属性で、中の要素を取れます。

In [28]:
el = plots.Overlay.II.Peak.Legendre_3
print(f"{el}, {el.group=}, {el.label=}")

:Scatter   [x]   (y), el.group='peak', el.label='legendre 3'


`plots` を出すときに、`opts()` で各要素の表示を決めます。特定の要素だけ変えたいときは、第 1 引数にその `group` と `label` を渡します。ここでは `text_hi` と `text_lo` の `Labels` を別々にしています。

In [29]:
plots.opts(
    opts.Layout(shared_axes=False),
    opts.Scatter(size=10, marker="x"),
    opts.Curve(alpha=0.7),
    opts.Overlay(show_grid=True, width=400, legend_cols=3),
    opts.Labels(text_font_size="9pt"),
    opts.Labels("text_hi", text_baseline="bottom"),
    opts.Labels("text_lo", text_baseline="top"),
)

:Layout
   .Overlay.I  :Overlay
      .Curve.Chebyu_2   :Curve   [x]   (y)
      .Peak.Chebyu_2    :Scatter   [x]   (y)
      .Text_hi.Chebyu_2 :Labels   [x,y]   (Label)
      .Text_lo.Chebyu_2 :Labels   [x,y]   (Label)
      .Curve.Chebyu_3   :Curve   [x]   (y)
      .Peak.Chebyu_3    :Scatter   [x]   (y)
      .Text_hi.Chebyu_3 :Labels   [x,y]   (Label)
      .Text_lo.Chebyu_3 :Labels   [x,y]   (Label)
      .Curve.Chebyu_4   :Curve   [x]   (y)
      .Peak.Chebyu_4    :Scatter   [x]   (y)
      .Text_hi.Chebyu_4 :Labels   [x,y]   (Label)
      .Text_lo.Chebyu_4 :Labels   [x,y]   (Label)
   .Overlay.II :Overlay
      .Curve.Legendre_2   :Curve   [x]   (y)
      .Peak.Legendre_2    :Scatter   [x]   (y)
      .Text_hi.Legendre_2 :Labels   [x,y]   (Label)
      .Text_lo.Legendre_2 :Labels   [x,y]   (Label)
      .Curve.Legendre_3   :Curve   [x]   (y)
      .Peak.Legendre_3    :Scatter   [x]   (y)
      .Text_hi.Legendre_3 :Labels   [x,y]   (Label)
      .Text_lo.Legendre_3 :Labels   [x,y]   (Label)
      .Curve.Legendre_4   :Curve   [x]   (y)
      .Peak.Legendre_4    :Scatter   [x]   (y)
      .Text_hi.Legendre_4 :Labels   [x,y]   (Label)
      .Text_lo.Legendre_4 :Labels   [x,y]   (Label)

### 入れ子にする

`UniformNdMapping` の派生は入れ子にでき、複雑な図を作れます。次の `run_sim()` は、質量・バネ・ダンパーの系の微分方程式を `solve_ivp()` で解き、`Curve` を返します。

In [30]:
from scipy import integrate
import numpy as np

def spring_sys(t, y, m, b, k, F):
    x, v = y
    dx = v
    dv = (F - k * x - b * v) / m
    return dx, dv

def spring_sim(x0, v0, m, b, k, F, tend, n):
    t = np.linspace(0, tend, n)
    res = integrate.solve_ivp(spring_sys, (0, tend), (x0, v0), t_eval=t, args=(m, b, k, F))
    return res.t, res.y[0], res.y[1]

dim_time = hv.Dimension('t', label='time', unit='s')
dim_pos = vdims=hv.Dimension('x', label='x', unit='m')

def run_sim(x0, v0, m, b, k, F, tend, n): 
    t, x, _ = spring_sim(x0, v0, m, b, k, F, tend, n)
    return hv.Curve((t, x), kdims=dim_time, vdims=dim_pos)

`np.mgrid()` で `m`、`b`、`k`、`F` の 4 次元の格子を作り、各組で `run_sim()` を呼んで `HoloMap` の `hm` にします。キー次元は 4 つで、各値が `Curve`(キー `t`、値 `x`)です。

In [31]:
from itertools import product

hm = hv.HoloMap(kdims=[
    hv.Dimension('m', label='mass', unit='kg'),
    hv.Dimension('b', label='dumper', unit=''),
    hv.Dimension('k', label='spring', unit='N/m'),
    hv.Dimension('F', label='force', unit='N'),
])

for args in np.mgrid[0.5:2:3j, 0.1:1.0:3j, 0.5:3:3j, 0:1:3j].reshape(4, -1).T:
    hm[tuple(args)] = run_sim(1.0, 0.0, *args, tend=10, n=50)

print(hm)

:HoloMap   [m,b,k,F]
   :Curve   [t]   (x)


`overlay()` と `layout()` をつなげて `NdLayout` にし、`cols()` で列の数、`opts()` で表示を決めます。`plots` は `m` と `b` の 2 次元の `NdLayout` で、各要素が `F` の `HoloMap`、その各要素が `k` の `NdOverlay` です。

In [32]:
options = [
    opts.NdOverlay(legend_position='top_left', legend_cols=3), 
    opts.Curve(show_grid=True, width=300)    
]    
    
plots = hm.overlay('k').layout(['m', 'b']).cols(3).opts(*options)
print(plots)

:NdLayout   [m,b]
   :HoloMap   [F]
      :NdOverlay   [k]
         :Curve   [t]   (x)


In [33]:
plots

:NdLayout   [m,b]
   :HoloMap   [F]
      :NdOverlay   [k]
         :Curve   [t]   (x)

表示すると、`NdLayout` が `m` と `b` を並べ、`NdOverlay` が `k` を 1 つの図に重ね、`HoloMap` が `F` をスライダーにします。

## プロットのオプション

HoloViews の要素と入れ物は、データと次元の関係を表すだけで、色や線種は持ちません。見た目は `opts` で決めます。`opts` のメソッド名は要素のクラス名で、キーワードで値を決めます。次の例は `Curve` と `Scatter` のオプションです。`width` が幅、`show_grid` がグリッド、`line_width` が線の太さ、`color` が色です。`group` と `label` も指定でき、`scatter_opt2` は `group` が `Large` のものだけに効きます。

`hv.help(hv.Curve)` で、使えるオプションの一覧と説明が出ます。

In [34]:
from holoviews import opts

curve_opt = opts.Curve(width=600, show_grid=True, line_width=4, color="green")
scatter_opt = opts.Scatter(size=16, marker="x", color="red")
scatter_opt2 = opts.Scatter("Large", size=24, color="blue")

`opts()` はそのオブジェクトに、`options()` は複製してから設定します。どちらも複数のオプションを受け取ります。`opts.info()` で設定を見られ、`show_defaults=True` なら既定値も出ます。`opts.clear()` で消せます。

設定はオブジェクトの中には入らず、HoloViews が全体の対応表で持ちます。

次のコードは `Overlay` を作り、2 つのオプションを渡します。

In [35]:
x = np.linspace(0, 4 * np.pi, 50)
y = np.sin(x)
data = x, y
data2 = x[::5], y[::5]
overlay = hv.Curve(data) * hv.Scatter(data) * hv.Scatter(data2, group="Large")
overlay.opts(curve_opt, scatter_opt, scatter_opt2)
overlay.opts.info()
print()
overlay.opts.clear()
overlay.opts.info(show_defaults=True)

:Overlay
   .Curve.I   :Curve   [x]   (y)
    | Options(color='green', line_width=4, show_grid=True, width=600)
   .Scatter.I :Scatter   [x]   (y)
    | Options(color='red', marker='x', size=16)
   .Large.I   :Scatter   [x]   (y)
    | Options(color='blue', marker='x', size=24)

:Overlay
 | Options(axiswise=False, click_policy='mute', framewise=False)
   .Curve.I   :Curve   [x]   (y)
    | Options(axiswise=False, color=Cycle(), framewise=False, line_width=2, muted_alpha=0.2)
   .Scatter.I :Scatter   [x]   (y)
    | Options(axiswise=False, cmap='kbc_r', color=Cycle(), framewise=False, muted_alpha=0.2,
    |         size=np.float64(2.449489742783178))
   .Large.I   :Scatter   [x]   (y)
    | Options(axiswise=False, cmap='kbc_r', color=Cycle(), framewise=False, muted_alpha=0.2,
    |         size=np.float64(2.449489742783178))


既定のオプションは `opts.defaults()` で決めます。この章の冒頭でも、`Scatter`、`Curve`、`NdOverlay` の既定を決めています。

```python
opts.defaults(
    opts.Scatter(tools=['hover'], width=500, show_grid=True),     
    opts.Curve(tools=['hover'], width=500, show_grid=True), 
    opts.NdOverlay(legend_position='left'),
)
```

`opts()` では、データの次元に合わせて見た目を決められます。`Points` は 2 次元の点で、既定ではキー次元が座標です。`opts()` で、大きさ、色、形を次元につなげられます。

次の例では、`life` と `ppp_capita` を座標に、`year` を色、`population` を大きさにします。次元を `dim` で包み、演算子やメソッドで計算します。ここでは `population` を `norm()` で 0〜1 にし、平方根を取り、10 を掛けて、小さい点が見えるように定数を足します。

In [36]:
from holoviews import dim

points = ds.to.points(
    kdims=["life", "ppp_capita"], vdims=["year", "population"], groupby=[]
).opts(
    opts.Points(
        color=dim("year"),
        size=dim("population").norm() ** 0.5 * 10 + 2,
        colorbar=True,
        cmap="Viridis",
        width=500,
        show_grid=True,
    )
)
points

:Points   [life,ppp_capita]   (year,population)

`dim` の式は計算の手順を表すだけで、その場では計算しません。図を出すときに計算されます。`apply()` を呼んだのと同じです。

In [37]:
size_expr = dim("population").norm() ** 0.5 * 10 + 2
size = size_expr.apply(ds)
print(size_expr)
print(f"{size.min() = }, {size.max() = }")

(((dim('population').norm())**0.5)*10)+2
size.min() = np.float64(2.0), size.max() = np.float64(12.0)


## まとめ

* `extension('bokeh')` で描画に使うライブラリを選ぶ。
* データは `Element`(`Curve`、`Scatter` など)で包み、`kdims` と `vdims` で次元を決める。
* 次元の情報は `Dimension`。変えるときは `redim` を使う(元は変えない)。
* 整然データは `Dataset` にして、`to.curve()` などで図にする。
* 入れ物は `HoloMap`(スライダー)、`NdLayout`(並べる)、`NdOverlay`(重ねる)、`Layout`(`+`)、`Overlay`(`*`)。
* 見た目は `opts` で決める。要素のクラス名のメソッドにキーワードで渡す。
* `dim` の式で、次元の値から色や大きさを計算できる。